# Hierarchical classification in one API call

A product catalog has dozens of categories, grouped into departments. Instead of one long list, classify in two steps: pick the department, then a category inside it. With `when`, both steps happen in one TypeLLM API call, and only the chosen department's categories are asked.

1. Write the category tree.
2. Build the questions from the tree.
3. Classify each product in one call.

Paste your TypeLLM API key in the cell below, then run all cells.

In [1]:
%pip install -q "typellm>=0.6.7"

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [3]:
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY)

PRODUCTS = [
    "Noise-cancelling over-ear headphones, 30-hour battery",
    "14-inch laptop, 16 GB RAM, 512 GB SSD",
    "Mirrorless camera body, 24 MP, 4K video",
    "Stainless steel French press, 34 oz",
    "Cast iron skillet, 12 inch, pre-seasoned",
    "Queen size cotton sheet set, 400 thread count",
    "Men's waterproof rain jacket with hood",
    "Women's high-rise straight leg jeans",
    "Leather ankle boots, side zip",
    "Two-person backpacking tent, 4 lb, waterproof rainfly",
    "Adjustable dumbbells, 5 to 52.5 lb",
    "Road bike helmet with MIPS",
    "Vitamin C serum, 1 oz",
    "Matte liquid lipstick, long-wear",
    "Sulfate-free shampoo for curly hair, 16 oz",
    "1000-piece jigsaw puzzle, world map",
    "Strategy board game for 2 to 4 players",
    "Remote control off-road truck, 1:16 scale",
    "Gel ink pens, 12 colors",
    "Ergonomic mesh office chair with lumbar support",
    "Label maker with QWERTY keyboard",
    "Grain-free dry dog food, 30 lb bag",
    "Self-cleaning cat litter box",
    "Aquarium filter for 20-gallon tanks",
]

## Write the category tree

Eight departments with eight categories each: 64 categories in all.

In [4]:
TREE = {
    "electronics": ["phones", "laptops", "audio", "cameras", "tv_video", "wearables", "smart_home", "accessories"],
    "home_kitchen": ["cookware", "coffee_tea", "small_appliances", "bedding", "furniture", "storage", "lighting", "decor"],
    "clothing": ["tops", "bottoms", "dresses", "outerwear", "shoes", "underwear", "activewear", "accessories"],
    "sports_outdoors": ["camping", "cycling", "fitness", "hiking", "water_sports", "team_sports", "fishing", "winter_sports"],
    "beauty": ["skincare", "makeup", "haircare", "fragrance", "nail_care", "shaving", "bath_body", "beauty_tools"],
    "toys_games": ["building_sets", "dolls", "puzzles", "board_games", "outdoor_play", "plush", "educational", "remote_control"],
    "office": ["writing", "paper", "desk_organization", "office_furniture", "printers", "labels", "binders", "calendars"],
    "pet_supplies": ["dog_food", "cat_food", "litter", "toys", "beds", "grooming", "aquarium", "health"],
}

## Build the questions from the tree

`department` chooses from the departments. Each department gets its own field with its categories, asked only `when` the department matches. However large the tree grows, this code stays the same.

In [5]:
def build_questions(tree):
    questions = {"department": {
        "type": "string", "enum": list(tree),
        "instructions": "Which department does this product belong to?",
    }}
    for department, categories in tree.items():
        questions[department] = {
            "type": "string", "enum": categories, "when": {"department": department},
            "instructions": f"Which {department.replace('_', ' ')} category does this product belong to?",
        }
    return questions

QUESTIONS = build_questions(TREE)

## Classify each product in one call

The answer holds the department and the one category field that ran.

In [6]:
def classify(product):
    result = client.generate(context=f"Product: {product}", questions=QUESTIONS).result
    department = result["department"]
    return department, result[department]

assigned = {}  # product -> (department, category)
for product in PRODUCTS:
    assigned[product] = classify(product)
    department, category = assigned[product]
    print(f"{department:16s} {category:18s} <- {product}")

electronics      audio              <- Noise-cancelling over-ear headphones, 30-hour battery


electronics      laptops            <- 14-inch laptop, 16 GB RAM, 512 GB SSD


electronics      cameras            <- Mirrorless camera body, 24 MP, 4K video


home_kitchen     coffee_tea         <- Stainless steel French press, 34 oz


home_kitchen     cookware           <- Cast iron skillet, 12 inch, pre-seasoned


...

office           office_furniture   <- Ergonomic mesh office chair with lumbar support


office           labels             <- Label maker with QWERTY keyboard


pet_supplies     dog_food           <- Grain-free dry dog food, 30 lb bag


pet_supplies     litter             <- Self-cleaning cat litter box


pet_supplies     aquarium           <- Aquarium filter for 20-gallon tanks


## Final result

In [7]:
for department in TREE:
    print(department)
    for category in TREE[department]:
        members = [p for p, (d, c) in assigned.items() if (d, c) == (department, category)]
        if members:
            print(f"  {category:18s} {', '.join(members)}")

electronics
  laptops            14-inch laptop, 16 GB RAM, 512 GB SSD
  audio              Noise-cancelling over-ear headphones, 30-hour battery
  cameras            Mirrorless camera body, 24 MP, 4K video
home_kitchen
  cookware           Cast iron skillet, 12 inch, pre-seasoned
  coffee_tea         Stainless steel French press, 34 oz
  bedding            Queen size cotton sheet set, 400 thread count
clothing
  bottoms            Women's high-rise straight leg jeans
  outerwear          Men's waterproof rain jacket with hood
  shoes              Leather ankle boots, side zip
sports_outdoors
  camping            Two-person backpacking tent, 4 lb, waterproof rainfly
  cycling            Road bike helmet with MIPS
  fitness            Adjustable dumbbells, 5 to 52.5 lb
beauty
  skincare           Vitamin C serum, 1 oz
  makeup             Matte liquid lipstick, long-wear
  haircare           Sulfate-free shampoo for curly hair, 16 oz
toys_games
  puzzles            1000-piece jigsaw puz